# Chapter 2 — A First Model of the Brain

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0 and 1.  
**Author:** Er. Rishabh Aryan

Chapter 0 supplied a minimum of anatomy. This chapter builds an *engineering model*: only those facts about neurons, synapses, maps, and fields that later chapters will use as premises. It is not a substitute for a neuroscience course. Where the model is known to be too simple, the text says so.

**Learning outcomes**

1. Describe a neuron as an integrator and a pulse generator, and a synapse as a plastic weight.
2. Explain why most BCI recordings are population fields rather than spike transcripts.
3. Name the cortical territories that motor, speech, and cognitive interfaces typically approach, and state what “typically” conceals.
4. Treat oscillations as statistics of populations, not as a codebook of thoughts.
5. State the implementational gap that cognitive computational neuroscience tries to close, and why a decoder can succeed without closing it.


---

## 2.1 Three levels, one measurement

Marr’s classic distinction is useful here if it is used lightly.

- **Computational level.** What problem is being solved? Reach a target. Select a letter. Hold an item in working memory.
- **Algorithmic level.** What representation and procedure implement that solution? An accumulator of evidence. A motor plan in joint coordinates. A phonological sequence.
- **Implementational level.** What physical variables in tissue carry those procedures? Spike times. Synaptic weights. Extracellular fields.

A BCI records at the implementational level and is asked, almost always, to recover something closer to the computational level: *what did the person intend?* The mapping is not one-to-one. Many microscopic arrangements produce similar fields. A decoder that works has found a usable statistical bridge. It has not thereby discovered the algorithm the brain is running.

Cognitive computing, in the sense of Chapter 1, lives mainly at the computational and algorithmic levels. It becomes part of a BCI when those levels are used to choose features, priors, or closed-loop policies. It does not replace the electrode.

> **Definition 2.1 (Engineering model of the brain).**  
> The model used in this book treats the cortex as a large set of electrically signalling cells, coupled by plastic synapses, organised in part as spatial maps, and generating extracellular fields that electrodes can sample. Higher cognition is admitted as a target of decoding only insofar as it leaves a statistical trace in those fields.


---

## 2.2 The neuron as a device

A neuron receives current-like influence through its dendrites, integrates that influence in the cell body, and, when a threshold is crossed, emits an **action potential**: a brief, stereotyped pulse lasting on the order of one millisecond. The shape of the pulse carries little information. Rate and timing carry more.

Two modelling remarks matter for BCI.

**Remark 1 — All-or-none output, graded input.**  
Dendritic and somatic potentials vary continuously. The outgoing spike does not. Population recordings (EEG, ECoG, LFP) are dominated by the graded **postsynaptic** currents of many cells, not by the spikes themselves. Intracortical microelectrodes can see both.

**Remark 2 — A cell is not a labelled line for a thought.**  
Some neurons in motor cortex vary their rate with intended direction. Some neurons in parietal cortex vary with goals, numbers, or intended effectors. Many neurons mix several variables in the same spike train (*mixed selectivity*). Finding a cell that correlates with a task is not the same as finding a cell that *means* the task.

A point-neuron cartoon is enough for this book:

$$
\tau \frac{dv}{dt} = -v(t) + \sum_j w_j s_j(t) + I_{\mathrm{ext}}(t),
$$

with a spike declared when $v$ crosses a threshold and is then reset. Here $w_j$ is a synaptic weight and $s_j$ is incoming activity. The equation is a sketch. Real dendrites have structure; real thresholds wander; real cells emit bursts. The sketch earns its place because it makes two later claims thinkable: a decoder can look at a weighted sum of inputs, and those weights can change.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(2)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})

def leaky_integrate_and_fire(T=2000, dt=0.001, tau=0.020, v_th=1.0, v_reset=0.0, I=1.20):
    # Minimal LIF: tau dv/dt = -v + I + noise. Spikes when I is above threshold.
    n = int(T)
    v = np.zeros(n)
    spikes = []
    for t in range(1, n):
        dv = (dt / tau) * (-v[t - 1] + I + 0.15 * np.random.randn())
        v[t] = v[t - 1] + dv
        if v[t] >= v_th:
            v[t] = v_reset
            spikes.append(t * dt)
    return np.arange(n) * dt, v, np.array(spikes)

t, v, spikes = leaky_integrate_and_fire()
print(f"emitted {spikes.size} spikes in {t[-1]:.2f} s; mean rate = {spikes.size / t[-1]:.1f} Hz")

fig, ax = plt.subplots(figsize=(7.4, 2.8))
ax.plot(t, v, color="steelblue", linewidth=0.9)
for s in spikes:
    ax.axvline(s, color="darkorange", linewidth=0.6, alpha=0.7)
ax.set_xlabel("time (s)")
ax.set_ylabel("v (a.u.)")
ax.set_title("Leaky integrate-and-fire cartoon (not a biological recording)")
plt.tight_layout()
plt.show()


The figure is a teaching device. An EEG trace will never look like this. An isolated extracellular spike on a microelectrode can look more like a brief pulse than like this subthreshold ramp. Keep the cartoon for the *logic* of integration and threshold; do not treat it as data.


---

## 2.3 Synapses and why the mapping drifts

A **synapse** is the junction at which one neuron influences another, usually chemically. In the point-neuron cartoon it is a weight $w_j$. In tissue it is a structured, metabolically expensive object that changes with use.

**Long-term potentiation and depression** alter weights over minutes to days. **Spike-timing-dependent plasticity** makes the sign of the change depend on the order of pre- and postsynaptic spikes. **Short-term** facilitation and depression alter effective weights over milliseconds to seconds.

Three engineering consequences follow.

1. **The user learns.** Amina can discover a mental strategy that moves a cursor more reliably this afternoon than this morning. That is not noise. It is the nervous system using the decoder as part of its world.
2. **The decoder must be allowed to adapt.** A weight matrix fitted on Monday is a hypothesis about Tuesday, not a law.
3. **Rehabilitation tries to aim plasticity.** Chapter 10 will treat pairing of decoded intent with movement or stimulation as an attempt to drive useful change in tissue, not merely to read a fixed code.

A BCI that treats the brain as a frozen transmitter will work in a short demonstration and then surprise its authors.


---

## 2.4 From one cell to a field

A pyramidal neuron in cortex is, electrically, a small battery in a conductive volume. Coordinated synaptic current in many aligned cells produces an extracellular voltage: a **field potential**.

| Distance from the generators | Name | Dominant content |
|---|---|---|
| Tens of micrometres | Multi-unit activity and LFP | Spikes of one or a few cells; local synaptic current |
| Millimetres, cortical surface | ECoG | Population fields, intermediate spatial grain |
| Centimetres, scalp | EEG | Heavily volume-conducted, skull-filtered mixture |

Two physical facts do a great deal of later work.

**Volume conduction.** Current flows through tissue and, for EEG, through skull and scalp. A scalp electrode is therefore not “over” a unique source. It sees a weighted sum of many sources, with weights that depend on geometry and conductivity.

**Cancellation.** Sources that are randomly oriented, or that form closed fields, contribute little at a distance. EEG is biased toward large, open, radially oriented populations near the surface. Deep and spatially salt-and-pepper codes are poorly visible from the scalp even when they are computationally important.

> **Definition 2.2 (What an electrode reports).**  
> An electrode reports a linear mixture of nearby electrical events, filtered by tissue and by the electrode’s size. It does not report a proposition.

This is the biological ground of Chapter 1’s refusal to say that a headset reads thoughts.


In [ ]:
# Volume conduction cartoon: four cortical "sources", one scalp electrode.
# Each source is a brief oscillation; the electrode sees a weighted sum.

fs = 256
t = np.arange(0, 1.0, 1 / fs)
sources = np.vstack([
    1.0 * np.sin(2 * np.pi * 10 * t),           # strong, radial, nearby
    0.7 * np.sin(2 * np.pi * 22 * t + 0.4),     # beta-like
    0.5 * np.sin(2 * np.pi * 6 * t + 1.1),      # deeper / weaker
    0.9 * np.sin(2 * np.pi * 50 * t),           # cancelled / poorly conducted
])
# mixing weights at the scalp (last source almost invisible)
a = np.array([0.9, 0.4, 0.25, 0.05])
eeg = a @ sources + 0.15 * np.random.randn(t.size)

fig, axes = plt.subplots(5, 1, figsize=(7.4, 6.2), sharex=True)
labels = ["source 1 (nearby 10 Hz)", "source 2", "source 3", "source 4 (poorly conducted)", "scalp mixture"]
series = [sources[0], sources[1], sources[2], sources[3], eeg]
colours = ["#4c6a92"] * 4 + ["#b85c38"]
for ax, s, lab, c in zip(axes, series, labels, colours):
    ax.plot(t, s, color=c, linewidth=0.9)
    ax.set_ylabel("a.u.")
    ax.set_title(lab, loc="left", fontsize=10)
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()
print("mixing weights a =", a)
print("The scalp trace is not identical to any one source.")


---

## 2.5 Maps an engineer may rely on

Cortex is not soup. It is, in part, a set of maps. The following list is a working atlas for this course, not a claim that function lives only here.

**Primary motor cortex (M1), precentral gyrus.**  
A coarse map of the body. Firing and fields here vary with intended movement. Classical motor BCIs record here. The map is plastic and overlapping; it is not a panel of labelled buttons.

**Primary somatosensory cortex (S1), postcentral gyrus.**  
A map of the body surface. Relevant when the interface must write touch rather than only read intent.

**Premotor and supplementary motor areas.**  
Planning, sequences, and some effector-independent goals. Useful when the scientific target is a plan rather than a pull.

**Posterior parietal cortex.**  
Visuospatial goals, attention, some numerical and effector-independent intention. Cognitive signals in implanted research often appear here as mixed selectivity, not as a single “thought neuron.”

**Speech-related cortex.**  
Superior temporal regions participate in acoustic and phonetic processing. Ventral motor and inferior frontal regions participate in articulation and higher-order structure. Attempted-speech BCIs record from some of these territories. Inner speech is not guaranteed to reuse the same pattern as attempted speech.

**Prefrontal cortex.**  
Control, working memory, value, and task sets. A tempting target for cognitive BCI and a difficult one: representations are distributed, context-dependent, and easy to over-interpret in a small sample of tasks.

**Subcortical structures.**  
Basal ganglia, thalamus, and hippocampus matter to disease and to deep-brain stimulation. Most introductory BCI recordings still come from cortex or from the scalp above it. Do not infer that cognition is a cortical surface phenomenon.

Amina’s communication systems, when non-invasive, see a blurred mixture biased toward surface maps. When implanted for attempted speech, they see a finer mixture from a chosen patch. Neither view is the language system.


---

## 2.6 Rhythms as population statistics

Oscillations are rhythmic fluctuations of population fields. Conventional bands are bookkeeping devices.

| Band | Approximate range | Use in this book |
|---|---|---|
| Delta | 0.5–4 Hz | Slow states; treat cognitive claims cautiously |
| Theta | 4–8 Hz | Memory and control literatures; not a switch |
| Alpha / mu | 8–13 Hz | Posterior idle alpha; sensorimotor mu |
| Beta | 13–30 Hz | Motor maintenance; effort-related claims need controls |
| Gamma | 30–80+ Hz | Local population engagement; at the scalp, often confounded |

**Event-related desynchronisation (ERD)** of mu/beta over sensorimotor cortex during motor imagery is a workhorse of non-invasive BCI. The physiology is a change in a population rhythm, visible as a drop in band power. It is not the appearance, on the scalp, of a spike code for “left hand.”

**Event-related potentials (ERPs)** such as P300 are time-locked deflections of the field after a stimulus. They are also population events. Averaging reveals them because trial-to-trial noise is large relative to the deflection.

Do not memorise a dictionary that maps each band to a cognitive faculty. The same band participates in many functions; the same function rides on more than one band. Features in Chapter 6 will use bands because they are convenient coordinates, not because they are natural kinds.


In [ ]:
fs = 256
t = np.arange(0, 2.0, 1 / fs)
# idle: strong mu; imagery: attenuated mu, slightly higher gamma-like jitter
mu_idle = 1.3 * np.sin(2 * np.pi * 10 * t)
mu_img = 0.35 * np.sin(2 * np.pi * 11 * t)
beta = 0.25 * np.sin(2 * np.pi * 20 * t)
noise = 0.3 * np.random.randn(t.size)
idle = mu_idle + beta + noise
imagery = mu_img + beta + 0.35 * np.random.randn(t.size)


def band_power(sig, band, fs=fs):
    spec = np.fft.rfft(sig * np.hanning(sig.size))
    freqs = np.fft.rfftfreq(sig.size, 1 / fs)
    psd = np.abs(spec) ** 2
    mask = (freqs >= band[0]) & (freqs < band[1])
    return psd[mask].mean()

print(f"mu power idle    = {band_power(idle, (8, 13)):.1f}")
print(f"mu power imagery = {band_power(imagery, (8, 13)):.1f}")

fig, axes = plt.subplots(2, 1, figsize=(7.4, 4.0), sharex=True)
axes[0].plot(t, idle, color="steelblue", linewidth=0.8)
axes[0].set_title("Cartoon sensorimotor trace: idle")
axes[0].set_ylabel("a.u.")
axes[1].plot(t, imagery, color="darkorange", linewidth=0.8)
axes[1].set_title("Cartoon sensorimotor trace: motor imagery (attenuated mu)")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("a.u.")
plt.tight_layout()
plt.show()


---

## 2.7 Coding: rate, timing, populations, mixing

Four ideas about neural code appear later and should be named now.

**Rate coding.** Information in how often a cell spikes in a window. Still the default assumption of many motor decoders.

**Temporal coding.** Information in when spikes occur relative to one another or to a rhythm. Harder to read from the scalp; sometimes visible intracortically.

**Population coding.** The relevant variable is carried by a pattern across cells, not by one labelled neuron. Almost every working BCI assumes this, even when it uses only a few features.

**Mixed selectivity.** A single cell (or a single channel) varies with more than one task variable. Mixed selectivity is a problem for storytelling (“the reach neuron”) and a resource for linear readout: a linearly separable combination of mixed cells can recover variables that no cell represents alone.

Cognitive BCI inherits the last two in an acute form. A working-memory item is not known to sit as a stable rate in one prefrontal channel. A decoder that classifies “load high versus low” from EEG has found a correlate of a coarse state. It has not recovered the contents of the buffer.


---

## 2.8 The implementational gap

Cognitive computational neuroscience tries to connect all three of Marr’s levels: to show that a proposed algorithm is actually implemented in measured neural activity. That programme is not identical with BCI.

A BCI may succeed while remaining silent about algorithms. A linear decoder that moves Amina’s cursor has solved an engineering problem. It has not thereby shown that motor cortex implements that linear readout. Conversely, a beautiful model of evidence accumulation may explain behaviour and still yield no usable electrode feature.

The book will use models in three disciplined ways:

1. as **sources of features** (for example band power where ERD is expected);
2. as **priors** (for example linguistic structure over intended words);
3. as **policies** (for example slowing the interface when inferred workload is high).

It will not use models as proof that the interface has understood a mind.

> **Definition 2.3 (Usable trace).**  
> A neural measurement is a usable trace of a cognitive or motor variable if a decoder, trained and tested without leakage, predicts that variable at a level that changes a user’s options. Correlation with a theory is not sufficient.


---

## 2.9 What to carry into Chapter 3

Chapter 3 places sensors and stimulators onto the anatomy of this chapter. Carry four sentences.

1. Spikes are the language of individual cells; fields are the language of populations; most BCIs speak the second language, some implants speak both.
2. Synapses change, so the mapping from intent to recording is a moving target.
3. Maps exist and are coarser than cartoons; speech, motor, and prefrontal targets are different engineering objects.
4. A decoder can be useful without being an explanation.

Amina does not need a complete theory of language to spell. She needs a trace that survives Tuesday.


---

## 2.10 Chapter summary

The engineering model of the brain used in this book is a population of electrically signalling cells, coupled by plastic synapses, organised in part as maps, and generating fields that electrodes sample as linear mixtures. Action potentials are all-or-none pulses; the currents that dominate EEG are graded and synaptic. Rhythms are statistics of those populations. Mixed selectivity and volume conduction guarantee that an electrode is not a labelled line. Cognitive computing addresses computational and algorithmic questions; BCI records at the implementational surface and must justify every inferred faculty as a usable trace.


---

## Exercises

**2.1** In four sentences, explain why averaging EEG trials can reveal a P300 that is invisible in one trial, and name the cost of that averaging for a live communication system.

**2.2** A journalist writes that an implant “records thoughts from a single neuron.” Rewrite the sentence so that it could appear in this book.

**2.3** Using Definition 2.3, decide whether each of the following, if true, would count as a usable trace: (a) mu power falls when Amina imagines moving her right hand; (b) a prefrontal EEG feature correlates with a working-memory questionnaire across a group; (c) an attempted-speech decoder produces correct words at a rate Amina accepts for conversation. One sentence each.

**2.4** Why might a decoder trained at 09:00 fail at 16:00 even if the cap has not moved? Give one synaptic or metabolic reason and one strategic reason.

**2.5** Distinguish rate coding, population coding, and mixed selectivity with a single concrete motor example.

**2.6** Which of Marr’s three levels does a P300 speller, as used by Amina, primarily exploit? Which level does it leave untouched? Argue in 200 words.

**2.7** Sketch (on paper or in a notebook cell) four sources and one electrode, assign mixing weights, and state which source could vanish from the scalp recording without vanishing from the brain.


---

## Annotated reading

- Kandel et al., *Principles of Neural Science* — chapters on the neuron and synaptic plasticity; consult, do not attempt cover to cover before Chapter 3.  
- Buzsáki, G. *Rhythms of the Brain* — opinionated, useful corrective to band-as-faculty dictionaries.  
- Marr, D. *Vision* — the three-level distinction, to be used as a scaffold, not as dogma.  
- A current review of mixed selectivity in prefrontal and parietal cortex (instructor to nominate annually).

---

*End of Chapter 2.*  
*Next: Chapter 3 — Measuring and stimulating.*
